In [1]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [2]:
from models import Ride, ride_deserializer 

In [3]:
from kafka import KafkaConsumer
from models import ride_deserializer # Bring in your custom function!

# 1. Initialize the Consumer WITH the deserializer attached
consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    value_deserializer=ride_deserializer, # <--- This does the magic!
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id='postgres_writer_1'
)



In [4]:
import psycopg2

conn = psycopg2.connect(
    host='127.0.0.1',
    port=5433,
    database='postgres',
    user='postgres',
    password='pass'
)
conn.autocommit = True
cur = conn.cursor()

In [18]:
print(f"Listening to green-trips and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value  # Fully formed Ride dataclass
    
    cur.execute(
        """INSERT INTO processed_events
           (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, 
            passenger_count, trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (
            ride.lpep_pickup_datetime,
            ride.lpep_dropoff_datetime,
            ride.PULocationID, 
            ride.DOLocationID,
            ride.passenger_count,
            ride.trip_distance, 
            ride.tip_amount,
            ride.total_amount
        )
    )
    
    count += 1
    if count % 100 == 0:
        conn.commit() # Save the batch!
        print(f"Inserted {count} rows...")

# Catch any leftover rows
conn.commit() 
consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...


KeyboardInterrupt: 

In [5]:
import psycopg2
from kafka import KafkaConsumer
from models import ride_deserializer

# 1. Rebuild the Database Table (Guarantees the 8 columns exist)
conn = psycopg2.connect(host='127.0.0.1', port=5433, database='postgres', user='postgres', password='pass')
conn.autocommit = True
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS processed_events;")
cur.execute("""
CREATE TABLE processed_events (
    lpep_pickup_datetime VARCHAR,
    lpep_dropoff_datetime VARCHAR,
    PULocationID INTEGER,
    DOLocationID INTEGER,
    passenger_count INTEGER,
    trip_distance DOUBLE PRECISION,
    tip_amount DOUBLE PRECISION,
    total_amount DOUBLE PRECISION
);
""")
print("Postgres table 'processed_events' rebuilt successfully.")

# 2. Fresh Consumer with a Brand New Group ID and Auto-Stop
consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    value_deserializer=ride_deserializer,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id='ultimate_postgres_writer_v3', # <--- The new lock-breaker
    consumer_timeout_ms=5000 # <--- Automatically stops the cell when finished!
)

# 3. Read and Insert
print("Listening to green-trips and writing to PostgreSQL...")
count = 0

for message in consumer:
    ride = message.value
    
    cur.execute(
        """INSERT INTO processed_events
           (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, 
            passenger_count, trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.lpep_pickup_datetime, ride.lpep_dropoff_datetime, ride.PULocationID, 
         ride.DOLocationID, ride.passenger_count, ride.trip_distance, 
         ride.tip_amount, ride.total_amount)
    )
    
    count += 1
    if count % 1000 == 0:
        print(f"Inserted {count} rows...")

print(f"Finished! Successfully wrote {count} rows to Postgres.")

Postgres table 'processed_events' rebuilt successfully.
Listening to green-trips and writing to PostgreSQL...
Inserted 1000 rows...
Inserted 2000 rows...
Inserted 3000 rows...
Inserted 4000 rows...
Inserted 5000 rows...
Inserted 6000 rows...
Inserted 7000 rows...
Inserted 8000 rows...
Inserted 9000 rows...
Inserted 10000 rows...
Inserted 11000 rows...
Inserted 12000 rows...
Inserted 13000 rows...
Inserted 14000 rows...
Inserted 15000 rows...
Inserted 16000 rows...
Inserted 17000 rows...
Inserted 18000 rows...
Inserted 19000 rows...
Inserted 20000 rows...
Inserted 21000 rows...
Inserted 22000 rows...
Inserted 23000 rows...
Inserted 24000 rows...
Inserted 25000 rows...
Inserted 26000 rows...
Inserted 27000 rows...
Inserted 28000 rows...
Inserted 29000 rows...
Inserted 30000 rows...
Inserted 31000 rows...
Inserted 32000 rows...
Inserted 33000 rows...
Inserted 34000 rows...
Inserted 35000 rows...
Inserted 36000 rows...
Inserted 37000 rows...
Inserted 38000 rows...
Inserted 39000 rows...
In